Zadanie 1

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .master("local[2]") \
    .appName("Pan Tadeusz Analysis") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.showConsoleProgress", "true") \
    .config("spark.executor.logs.rolling.enableCompression", "true") \
    .config("spark.executor.logs.rolling.strategy", "size") \
    .config("spark.executor.logs.rolling.maxSize", "10m") \
    .getOrCreate()

sc = spark.sparkContext
pan_tadeusz_file = sc.textFile("pan-tadeusz.txt")
lines_with_tadeusz = pan_tadeusz_file.filter(lambda line: "Tadeusz" in line).count()
print(f"Liczba linii zawierających słowo 'Tadeusz': {lines_with_tadeusz}")

Zadanie 2

In [4]:
longest_lines = pan_tadeusz_file.map(lambda line: (len(line), line)) \
    .top(3, key=lambda x: x[0])

print("Trzy najdłuższe linie:")
for length, line in longest_lines:
    print(f"({length} znaków): {line}")

Zadanie 3

In [2]:
import re
import json
stopwords_url = "https://raw.githubusercontent.com/bieli/stopwords/master/polish.stopwords.txt"
import requests
stopwords = set(requests.get(stopwords_url).text.splitlines())

def clean_text(line):
    return re.findall(r'\b\w+\b', line.lower())

unique_words_rdd = (
    pan_tadeusz_file
    .flatMap(clean_text)
    .filter(lambda word: word not in stopwords)
    .map(lambda word: (word, 1))
    .reduceByKey(lambda a, b: a + b)
)

word_counts = unique_words_rdd.collectAsMap()

with open("pan_tadeusz_bag_of_words.json", "w", encoding="utf-8") as json_file:
    json.dump(word_counts, json_file, ensure_ascii=False, indent=4)

most_frequent_word = max(word_counts.items(), key=lambda x: x[1])
print(f"Najczęściej występujące słowo: '{most_frequent_word[0]}' z liczbą wystąpień: {most_frequent_word[1]}")